# 03 — Hybrid Model Evaluation

Compares DBSCAN-only, classifier-only, and the fused hybrid score.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sklearn.metrics import ConfusionMatrixDisplay, average_precision_score, confusion_matrix

from intelligence.pipeline import config
from intelligence.pipeline.hybrid import HybridFraudDetector
from intelligence.pipeline.preprocessing import Preprocessor

preprocessor = Preprocessor()
train_df, score_df = preprocessor.prepare()

model = HybridFraudDetector().fit(train_df)
scored = model.score(score_df)
scored[["dbscan_score", "classifier_score", "risk_score", "risk_tier"]].head()

In [ ]:
y_true = scored[config.LABEL_COLUMN]

for name, col in [("DBSCAN-only", "dbscan_score"), ("Classifier-only", "classifier_score"), ("Hybrid", "risk_score")]:
    pr_auc = average_precision_score(y_true, scored[col])
    print(f"{name:16s} PR-AUC: {pr_auc:.4f}")

In [ ]:
y_pred = (scored["risk_score"] >= config.ALERT_THRESHOLD).astype(int)
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["normal", "fraud"]).plot()